# A First End-to-End Pipeline

**Week 1 · Lesson 8** &nbsp;·&nbsp; GBUS 564: Predictive Analytics &nbsp;·&nbsp; Fall 2026

<a target="_blank" href="https://colab.research.google.com/github/avorybryant/umw_gbus564_fall26/blob/main/notebooks/week1/lectures/08_first_end_to_end_pipeline.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

## Setup

Every notebook this term opens in Google Colab and runs the same handful of
libraries: **numpy** and **pandas** for data, **matplotlib** for plots, and
**scikit-learn** for models. The cell below is that setup, unchanged — you'll
see it verbatim at the top of every lesson notebook all semester:

- `%pip install -q ...` — installs this course's helper package (the dataset
  loaders and self-check functions every lesson imports) from the course's
  GitHub repo. Colab already has the core libraries, so it takes seconds.
- `set_config(display="diagram")` — makes scikit-learn pipelines render as
  a diagram instead of a text dump when displayed.
- `RANDOM_SEED = 564` — every random operation in this course (train/test
  splits, synthetic data, model initialization) is seeded with this same
  number, so results are reproducible run to run and comparable across
  lessons.

In [ ]:
# Environment setup — identical across every notebook in the course.
# Installs the course package (dataset loaders + helpers) from the public
# course repo; Colab already has numpy/pandas/scikit-learn, so it's quick.
%pip install -q matplotlib git+https://github.com/avorybryant/umw_gbus564_fall26.git

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import set_config

set_config(display="diagram")
plt.rcParams["figure.figsize"] = (8, 5)

RANDOM_SEED = 564  # GBUS 564 — reuse this seed everywhere for reproducibility
rng = np.random.default_rng(RANDOM_SEED)

## 1. Business Problem

**Why this matters:** every lesson this term reuses one notebook shape —
**Business Problem → Data → Model → Evaluate → Interpret → Recommend** — and
one Python stack running in Colab. Learn the mechanics once, here, and every
later notebook is just filling in the lesson-specific pieces: a real dataset,
a real model, a real business question. Skip this step and you re-learn
environment setup every week, and the tooling crowds out the actual content.

**Business question:** which customers are likely to cancel their
subscription next billing cycle, and which of those should get a retention
offer?

**Why it matters:** acquiring a new customer costs far more than keeping an
existing one. Missing an at-risk, high-value customer is lost revenue;
offering everyone a retention discount wastes budget on customers who were
never going to leave. A model that ranks customers by cancellation risk lets
the retention team target the offer instead of guessing.

This notebook walks the *whole* pipeline once, start to finish, on a
black-box model — the point isn't the model choice (that's Weeks 3–6), it's
seeing every stage of the map connect before we slow down and study each
stage in depth.

## 2. Data

*Load, describe, and sanity-check the data before touching a model.*

This is the shape you'll see in Section 2 of every notebook: load into a
`pandas.DataFrame`, `.head()` to eyeball it, `.describe()` to sanity-check
ranges and spot anything obviously wrong (missing values, impossible
values, a column that's secretly a string of numbers).

This lesson's real dataset is **Telco Customer Churn** — one row per
customer, with account/contract/service features and a `Churn` label — the
anchor dataset that carries this course through Week 6. It's committed as a
static file (`data/telco_churn/`, see `course_prep/course_data.md`) rather
than fetched over HTTPS, so it loads the same way every run. `load_churn()`
lives in `gbus564_coursework.utils.course_data`, shared across every lesson that uses this
dataset.

In [ ]:
from gbus564_coursework.utils.course_data import load_churn

df = load_churn()
df.head()

In [ ]:
df.describe(include="all")

*Fix what's wrong with the raw data, then turn what a retention analyst
already knows into a column the model can use.*

**Cleaning.** Two columns in the raw file don't arrive the way they should,
and `load_churn()` fixes both before you ever see them:

- `TotalCharges` is stored as *text*, because 11 brand-new customers
  (`tenure` of 0) have a blank instead of a number — they haven't been billed
  yet. Left alone, the column can't be used as a number at all, and the
  pipeline either crashes or treats every dollar amount as a category.
  The loader converts it to numeric and fills those 11 blanks with **0** (no
  bill yet — filling them with a month's charge would invent a payment that
  never happened). The cell below confirms it.
- `SeniorCitizen` is stored as `0`/`1`, while every other yes/no column is
  `"Yes"`/`"No"` text. It's a category, not a quantity — left as `0`/`1`,
  `.describe()` would report a "mean" for it and the pipeline would treat it
  as a number — so the loader recodes it to `"Yes"`/`"No"` to match.

In [ ]:
# TotalCharges: numeric, nothing missing, and the tenure-0 customers sit at 0.
print(f"TotalCharges dtype: {df['TotalCharges'].dtype}, missing: {df['TotalCharges'].isna().sum()}")
print(f"Customers with tenure 0: {(df['tenure'] == 0).sum()}, "
      f"their TotalCharges: {df.loc[df['tenure'] == 0, 'TotalCharges'].unique()}")
print(f"SeniorCitizen values: {sorted(df['SeniorCitizen'].unique())}")

**Feature engineering.** Three columns each mark a group that churns at
roughly 40–45%: a **month-to-month** contract, **fiber-optic** internet, and
paying by **electronic check**. Customers with all three at once are the
highest-risk cohort in this dataset, so we name that cohort explicitly with a
`high_risk_segment` flag — the step where business knowledge enters the
pipeline.

In [ ]:
df["high_risk_segment"] = (
    (df["Contract"] == "Month-to-month")
    & (df["InternetService"] == "Fiber optic")
    & (df["PaymentMethod"] == "Electronic check")
).map({True: "Yes", False: "No"})  # Yes/No text, like the dataset's other flags

df.groupby("high_risk_segment")["Churn"].agg(churn_rate="mean", customers="size").round(2)

Fewer than 1 in 5 customers fall in this segment, but they churn at about
60% versus roughly 19% for everyone else — together they account for over 40%
of all churners. Because `high_risk_segment` is a
Yes/No column, the encoding step defined next handles it like the rest.

**Hold out the test set, then define the encoding.** Two last steps
before any model sees the data:

- **Split first.** `train_test_split` sets aside 20% of customers the model
  never trains on, so Section 4 measures how it does on *new* customers, not
  how well it memorized old ones. `stratify=y` keeps the churn rate the same
  in both halves.
- **Define the encoding — don't apply it yet.** Churn has categorical
  columns (`Contract`, `PaymentMethod`, ...) that a model can't consume as
  text, so `OneHotEncoder` turns each category into its own 0/1 column; the
  numeric columns pass through unchanged. An encoder *learns* from whatever
  data it's fitted on (which categories exist), so it must only ever see
  training rows. That's why this cell only *defines* it — Section 3 fits it,
  inside the model's pipeline, on the training data alone.

The rule for every notebook this term: **define in Data, fit in Model, on
training data only.**

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

X, y = df.drop(columns="Churn"), df["Churn"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)

categorical_cols = X.select_dtypes(exclude="number").columns
preprocess = ColumnTransformer(
    [("encode", OneHotEncoder(handle_unknown="ignore", drop="if_binary"), categorical_cols)],
    remainder="passthrough",  # numeric columns pass through unchanged
)

print(f"Training customers: {len(X_train):,}   Test customers: {len(X_test):,}")

## 3. Model

*Baseline first — the reference every model must beat — then the model
under study.*

`DummyClassifier` (always predict the majority class) is the baseline: if a
real model can't beat that, it isn't adding value. The model itself is a
**black box** on purpose — a random forest, used here without tuning or a
deep look inside — because this notebook's job is the pipeline shape, not
the algorithm. Weeks 3–6 open the box (logistic regression, kNN,
trees/SHAP, neural nets) one model family at a time.

The model is a small `Pipeline`: the encoding step defined in Data Prep,
then the random forest. Calling `.fit()` on the pipeline fits both on the
training rows only — the encoder learns its categories from `X_train`,
never from the test set. Wrapping it as a pipeline — rather than encoding by
hand — is also what `set_config(display="diagram")` in Setup is for: it
renders the whole thing as a diagram wherever this cell's output is shown.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)

model = Pipeline([("prep", preprocess), ("clf", RandomForestClassifier(random_state=RANDOM_SEED))])
model.fit(X_train, y_train)

## 4. Evaluate

*Score against the baseline, not in isolation — a metric only means
something next to what it's beating.*

Plain accuracy is a coarse metric for a churn problem (churners are usually
the minority class) — Week 2 covers precision/recall/ROC-AUC and why
accuracy alone can mislead. Here it's enough to confirm the black-box model
is doing real work before we move on.

In [ ]:
from sklearn.metrics import accuracy_score

baseline_acc = accuracy_score(y_test, baseline.predict(X_test))
model_acc = accuracy_score(y_test, model.predict(X_test))

print(f"Baseline accuracy: {baseline_acc:.2%}")
print(f"Model accuracy:    {model_acc:.2%}")

## 5. Interpret

*What is the model actually keying off of? Business-readable, not just
statistically significant.*

A black-box model like a random forest has no coefficients to read — but it
does expose **feature importances**: how much each feature reduced impurity
across the forest's splits. That's a coarser signal than the SHAP values
Week 5 introduces, but it's enough to sanity-check that the model is keying
off features a retention analyst would recognize, not noise. Because the
categoricals were one-hot encoded inside the pipeline, the importances are
read off the pipeline's expanded feature names, not the original columns.

In [ ]:
feature_names = model.named_steps["prep"].get_feature_names_out()
importances = pd.Series(model.named_steps["clf"].feature_importances_, index=feature_names)
importances.sort_values().tail(10).plot.barh(title="Top features driving the prediction")
plt.tight_layout()

## 6. Recommend

**Recommendation:** rank customers by predicted cancellation risk and route
the top decile to the retention team for a proactive offer, rather than
offering discounts blanket-wide or only reacting after a cancellation
request comes in. The top importances above — tenure, contract type, total
and monthly charges — line up with what a retention analyst would already
suspect: new, month-to-month, higher-paying customers churn most. The
engineered `high_risk_segment` flag ranks right alongside contract type, which
gives the retention team a named segment to target, not just a ranked list.

**The rule:** every lesson notebook this term follows this same shape — the
same six headings, the same Setup cell, the same baseline-before-model
habit. Once you can read this notebook end-to-end, you can read every other
one this semester without relearning the scaffolding — what changes lesson
to lesson is the data, the model, and the business question, never the map.

**Caveats / next steps:** this run used a default-parameter model and plain
accuracy, so treat the number as a first pass, not a final answer. The
engineered flag didn't change this random forest's accuracy — a forest can
discover combinations like "month-to-month *and* fiber *and* electronic check"
on its own — so its payoff here is readability; simpler models in Week 3
benefit more directly. Week 2
swaps in the right metric for an imbalanced label, and Weeks 3–6 swap in
models that can be interpreted (and tuned) properly on this same dataset.